## Cell 1 — Install Dependencies



In [ ]:
!pip install ultralytics roboflow -q

Cell 2 — Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")   # replace before committing
project = rf.workspace("sabreena-akbar-s-workspace").project("meter-detection-n30cm")
version = project.version(4)
dataset = version.download("yolov8", location="/kaggle/working/dataset")

print("✅ Downloaded to:", dataset.location)

Cell 3 — Fix data.yaml Paths

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path("/kaggle/working/dataset/data.yaml")

data = yaml.safe_load(yaml_path.read_text())
data["path"]  = str(yaml_path.parent)
data["train"] = "train/images"
data["val"]   = "valid/images"
data["test"]  = "test/images"

yaml_path.write_text(yaml.safe_dump(data))
print("✅ data.yaml fixed")
print(open(yaml_path).read())

Cell 4 — Verify Dataset Split

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/dataset")
for split in ["train", "valid", "test"]:
    n = len(list((base / split / "images").glob("*")))
    print(f"{split}: {n} images")

Cell 5 — Train YOLOv8

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name="meter_yolov8n"
)

Cell 6 — Validate on Test Set

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/meter_yolov8n/weights/best.pt")
metrics = model.val(data=str(yaml_path), split="test")

print("Test mAP50:   ", metrics.box.map50)
print("Test mAP50-95:", metrics.box.map)

Cell 7 — Copy Best Model to a Simple Location

In [ ]:
import shutil
shutil.copy(
    "/kaggle/working/runs/detect/meter_yolov8n/weights/best.pt",
    "/kaggle/working/best.pt"
)
print("✅ best.pt copied to /kaggle/working/best.pt")

Cell 8 — Crop Test Images

In [ ]:
import cv2, json
from pathlib import Path
from ultralytics import YOLO

model = YOLO("/kaggle/working/best.pt")

INPUT_DIR  = Path("/kaggle/working/dataset/test/images")
OUTPUT_DIR = Path("/kaggle/working/meter_crops")
OUTPUT_DIR.mkdir(exist_ok=True)

log = []
for img_path in sorted(list(INPUT_DIR.glob("*.jpg")) + list(INPUT_DIR.glob("*.png"))):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    h, w = img.shape[:2]
    results = model(img, conf=0.5, verbose=False)[0]

    if len(results.boxes) == 0:
        print(f"⚠️  no meter: {img_path.name}")
        continue

    x1, y1, x2, y2 = map(int, results.boxes[0].xyxy[0])

    # add 5% margin
    mx, my = int((x2-x1)*0.05), int((y2-y1)*0.05)
    x1, y1 = max(0, x1-mx), max(0, y1-my)
    x2, y2 = min(w, x2+mx), min(h, y2+my)

    cv2.imwrite(str(OUTPUT_DIR / img_path.name), img[y1:y2, x1:x2])
    log.append({
        "file": img_path.name,
        "box":   [x1, y1, x2, y2],
        "conf":  float(results.boxes[0].conf[0])
    })

json.dump(log, open(OUTPUT_DIR / "boxes.json", "w"), indent=2)
print(f"\n✅ Cropped {len(log)} images → {OUTPUT_DIR}")